# TP 3 - Building a QSAR Model with Python

**Practical Cheminformatics**  
6th Semester - Applied Chemistry  
Duration: ~4 h

- Full name:
- Apogee code:
- Group:

## Objectives
By the end of this session you will be able to:
- Understand the QSAR workflow from data to prediction
- Download and curate a real bioactivity dataset from ChEMBL
- Compute molecular descriptors and fingerprints
- Split data into training and test sets
- Build a linear regression and a Random Forest model
- Evaluate model performance using R² and RMSE
- Interpret the model and understand the Applicability Domain

## Part 1 - The QSAR Pipeline Overview
QSAR (Quantitative Structure-Activity Relationships) connects molecular structure to biological activity:

**Activity = f(Molecular Descriptors)**

Pipeline:
1. Data collection
2. Data curation
3. Molecular representation
4. Model building
5. Model evaluation
6. Prediction

## Part 2 - Setting Up the Environment
### Step 2: Install required libraries

In [ ]:
!pip install rdkit chembl_webresource_client pandas scikit-learn matplotlib seaborn

### Step 3: Import libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cheminformatics
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, Draw, MACCSkeys
from rdkit import DataStructs

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# ChEMBL data access
from chembl_webresource_client.new_client import new_client

print("All libraries imported successfully! ✓")

**Q1. Did all imports work without errors?**

Your answer: ...

## Part 3 - Data Collection from ChEMBL
Target: **Acetylcholinesterase (CHEMBL220)**

### Step 4: Download bioactivity data

In [ ]:
activity = new_client.activity

results = activity.filter(
    target_chembl_id='CHEMBL220',
    standard_type='IC50',
    standard_relation='=',
    standard_units='nM'
).only([
    'molecule_chembl_id',
    'canonical_smiles',
    'standard_value',
    'standard_units',
    'standard_type'
])

df = pd.DataFrame.from_records(results)
print(f"Downloaded {len(df)} bioactivity records")
print(df.head())

**Q2. How many bioactivity records did you download?**

Your answer: ...

**Q3. Why filter for `standard_relation='='` and `standard_units='nM'`?**

Your answer: ...

### Step 5: Curate the data

In [ ]:
df = df.dropna(subset=['canonical_smiles', 'standard_value'])
print(f"After removing missing values: {len(df)} records")

df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
df = df.dropna(subset=['standard_value'])

df = df[(df['standard_value'] > 0.01) & (df['standard_value'] <= 100000)]
print(f"After removing extreme values: {len(df)} records")

df['pIC50'] = -np.log10(df['standard_value'] * 1e-9)
print(f"\npIC50 range: {df['pIC50'].min():.2f} to {df['pIC50'].max():.2f}")

df_clean = df.groupby('canonical_smiles').agg(
    molecule_chembl_id=('molecule_chembl_id', 'first'),
    pIC50=('pIC50', 'median'),
    n_measurements=('pIC50', 'count')
).reset_index()

print(f"After removing duplicates: {len(df_clean)} unique molecules")
print(df_clean.head())

**Q4. Why convert IC50 to pIC50?**

Your answer: ...

**Q5. Why use median instead of mean for duplicates?**

Your answer: ...

### Step 6: Visualise activity distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_clean['pIC50'], bins=30, color='steelblue', edgecolor='black')
axes[0].set_xlabel('pIC50')
axes[0].set_ylabel('Number of compounds')
axes[0].set_title('Distribution of pIC50')
axes[0].axvline(
    x=df_clean['pIC50'].mean(),
    color='red',
    linestyle='--',
    label=f"Mean = {df_clean['pIC50'].mean():.2f}"
)
axes[0].legend()

axes[1].boxplot(df_clean['pIC50'], vert=True)
axes[1].set_ylabel('pIC50')
axes[1].set_title('Box Plot of pIC50')

plt.tight_layout()
plt.show()

print(f"Mean pIC50: {df_clean['pIC50'].mean():.2f}")
print(f"Std pIC50:  {df_clean['pIC50'].std():.2f}")

**Q6. Describe the distribution. Is it roughly normal? Are there outliers?**

Your answer: ...

## Part 4 - Computing Molecular Descriptors
### Step 7: Compute descriptors

In [ ]:
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        'MW': Descriptors.MolWt(mol),
        'LogP': Descriptors.MolLogP(mol),
        'HBD': Descriptors.NumHDonors(mol),
        'HBA': Descriptors.NumHAcceptors(mol),
        'TPSA': Descriptors.TPSA(mol),
        'RotBonds': Descriptors.NumRotatableBonds(mol),
        'AromaticRings': Descriptors.NumAromaticRings(mol),
        'HeavyAtoms': Descriptors.HeavyAtomCount(mol),
        'RingCount': Descriptors.RingCount(mol),
        'FractionCSP3': Descriptors.FractionCSP3(mol)
    }

desc_list = []
valid_indices = []
for idx, row in df_clean.iterrows():
    desc = compute_descriptors(row['canonical_smiles'])
    if desc is not None:
        desc_list.append(desc)
        valid_indices.append(idx)

df_desc = pd.DataFrame(desc_list, index=valid_indices)
df_model = df_clean.loc[valid_indices].copy()
df_model = pd.concat([df_model.reset_index(drop=True), df_desc.reset_index(drop=True)], axis=1)

print(f"Computed descriptors for {len(df_model)} molecules")
print("\nDescriptor summary:")
print(df_model[['MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds']].describe().round(2))

**Q7. How many descriptors were computed per molecule? List them.**

Your answer: ...

### Step 8: Descriptor vs activity plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
descriptors_to_plot = ['MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds']

for ax, desc in zip(axes.flatten(), descriptors_to_plot):
    ax.scatter(df_model[desc], df_model['pIC50'], alpha=0.3, s=10, color='steelblue')
    ax.set_xlabel(desc)
    ax.set_ylabel('pIC50')
    ax.set_title(f'pIC50 vs {desc}')

plt.tight_layout()
plt.show()

**Q8. Which descriptors seem most correlated or uncorrelated with pIC50?**

Your answer: ...

### Step 9: Correlation matrix

In [ ]:
desc_cols = ['MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds', 'AromaticRings', 'HeavyAtoms', 'RingCount', 'FractionCSP3']
corr = df_model[desc_cols + ['pIC50']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Correlation Matrix: Descriptors & pIC50')
plt.tight_layout()
plt.show()

**Q9. Which descriptor pairs are highly correlated (|r| > 0.8), and why can this be problematic?**

Your answer: ...

## Part 5 - Building the QSAR Model
### Step 10: Prepare features and split data

In [ ]:
feature_cols = ['MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds', 'AromaticRings', 'HeavyAtoms']
X = df_model[feature_cols].values
y = df_model['pIC50'].values

print(f"Dataset size: {X.shape[0]} molecules, {X.shape[1]} features")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} molecules")
print(f"Test set:     {X_test.shape[0]} molecules")

**Q10. Why split into train/test sets? What happens if you evaluate on training data only?**

Your answer: ...

### Step 11: Model 1 - Linear Regression

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_train_pred_lr = lr_model.predict(X_train)
y_test_pred_lr = lr_model.predict(X_test)

r2_train_lr = r2_score(y_train, y_train_pred_lr)
r2_test_lr = r2_score(y_test, y_test_pred_lr)
rmse_train_lr = np.sqrt(mean_squared_error(y_train, y_train_pred_lr))
rmse_test_lr = np.sqrt(mean_squared_error(y_test, y_test_pred_lr))

print('=== Linear Regression Results ===')
print(f"Training:  R² = {r2_train_lr:.3f},  RMSE = {rmse_train_lr:.3f}")
print(f"Test:      R² = {r2_test_lr:.3f},  RMSE = {rmse_test_lr:.3f}")

print('\nModel coefficients:')
for feat, coef in zip(feature_cols, lr_model.coef_):
    print(f"  {feat:<15} : {coef:>8.4f}")
print(f"  {'Intercept':<15} : {lr_model.intercept_:>8.4f}")

**Q11. What is test R²? Is this a good model? What does R² represent?**

Your answer: ...

**Q12. Which descriptor has the largest absolute coefficient?**

Your answer: ...

### Step 12: Predicted vs actual (Linear Regression)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_train, y_train_pred_lr, alpha=0.3, s=10, color='steelblue')
axes[0].plot([3, 11], [3, 11], 'r--', linewidth=2)
axes[0].set_xlabel('Actual pIC50')
axes[0].set_ylabel('Predicted pIC50')
axes[0].set_title(f'Linear Regression — Training (R² = {r2_train_lr:.3f})')

axes[1].scatter(y_test, y_test_pred_lr, alpha=0.3, s=10, color='darkorange')
axes[1].plot([3, 11], [3, 11], 'r--', linewidth=2)
axes[1].set_xlabel('Actual pIC50')
axes[1].set_ylabel('Predicted pIC50')
axes[1].set_title(f'Linear Regression — Test (R² = {r2_test_lr:.3f})')

plt.tight_layout()
plt.show()

**Q13. Do points follow the diagonal line? Where does the model fail most?**

Your answer: ...

### Step 13: Model 2 - Random Forest

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

r2_train_rf = r2_score(y_train, y_train_pred_rf)
r2_test_rf = r2_score(y_test, y_test_pred_rf)
rmse_train_rf = np.sqrt(mean_squared_error(y_train, y_train_pred_rf))
rmse_test_rf = np.sqrt(mean_squared_error(y_test, y_test_pred_rf))

print('=== Random Forest Results ===')
print(f"Training:  R² = {r2_train_rf:.3f},  RMSE = {rmse_train_rf:.3f}")
print(f"Test:      R² = {r2_test_rf:.3f},  RMSE = {rmse_test_rf:.3f}")

**Q14. Compare LR vs RF performance (R² and RMSE). Which is better?**

| Metric | Linear Regression | Random Forest |
|---|---|---|
| R² (train) |  |  |
| R² (test) |  |  |
| RMSE (train) |  |  |
| RMSE (test) |  |  |

**Q15. Is there a large train-test gap for RF? What does it indicate?**

Your answer: ...

### Step 14: Predicted vs actual (Random Forest)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_train, y_train_pred_rf, alpha=0.3, s=10, color='steelblue')
axes[0].plot([3, 11], [3, 11], 'r--', linewidth=2)
axes[0].set_xlabel('Actual pIC50')
axes[0].set_ylabel('Predicted pIC50')
axes[0].set_title(f'Random Forest — Training (R² = {r2_train_rf:.3f})')

axes[1].scatter(y_test, y_test_pred_rf, alpha=0.3, s=10, color='darkorange')
axes[1].plot([3, 11], [3, 11], 'r--', linewidth=2)
axes[1].set_xlabel('Actual pIC50')
axes[1].set_ylabel('Predicted pIC50')
axes[1].set_title(f'Random Forest — Test (R² = {r2_test_rf:.3f})')

plt.tight_layout()
plt.show()

### Step 15: Feature importance

In [ ]:
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(len(feature_cols)), importances[indices], color='steelblue')
plt.xticks(range(len(feature_cols)), [feature_cols[i] for i in indices], rotation=45)
plt.ylabel('Feature Importance')
plt.title('Random Forest — Feature Importance')
plt.tight_layout()
plt.show()

print('Feature Importance Ranking:')
for i in indices:
    print(f"  {feature_cols[i]:<15} : {importances[i]:.4f}")

**Q16. Which descriptor is most important? Is it consistent with correlation analysis?**

Your answer: ...

## Part 6 - Model with Fingerprints (Optional Challenge)
### Step 16: Morgan fingerprints + Random Forest

In [ ]:
fp_list = []
valid_idx = []

for idx, row in df_model.iterrows():
    mol = Chem.MolFromSmiles(row['canonical_smiles'])
    if mol is not None:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
        arr = np.zeros((1024,), dtype=int)
        DataStructs.ConvertToNumpyArray(fp, arr)
        fp_list.append(arr)
        valid_idx.append(idx)

X_fp = np.array(fp_list)
y_fp = df_model.loc[valid_idx, 'pIC50'].values

print(f"Fingerprint matrix shape: {X_fp.shape}")
print(f"(Each molecule is represented by {X_fp.shape[1]} bits)")

X_fp_train, X_fp_test, y_fp_train, y_fp_test = train_test_split(
    X_fp, y_fp, test_size=0.2, random_state=42
)

rf_fp = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42)
rf_fp.fit(X_fp_train, y_fp_train)

y_fp_train_pred = rf_fp.predict(X_fp_train)
y_fp_test_pred = rf_fp.predict(X_fp_test)

r2_fp_train = r2_score(y_fp_train, y_fp_train_pred)
r2_fp_test = r2_score(y_fp_test, y_fp_test_pred)
rmse_fp_test = np.sqrt(mean_squared_error(y_fp_test, y_fp_test_pred))

print('\n=== Random Forest with Morgan Fingerprints ===')
print(f"Training:  R² = {r2_fp_train:.3f}")
print(f"Test:      R² = {r2_fp_test:.3f}, RMSE = {rmse_fp_test:.3f}")

**Q17. Compare descriptor-based and fingerprint-based models.**

| Model | Features | R² (test) | RMSE (test) |
|---|---|---|---|
| Linear Regression | Descriptors |  |  |
| Random Forest | Descriptors |  |  |
| Random Forest | Morgan FP (1024 bits) |  |  |

**Q18. Why might fingerprints perform differently from descriptors?**

Your answer: ...

## Part 7 - Applicability Domain
### Step 17: Determine AD using distance to centroid

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

centroid = X_train_scaled.mean(axis=0)

train_distances = np.sqrt(np.sum((X_train_scaled - centroid) ** 2, axis=1))
threshold = np.percentile(train_distances, 95)

test_distances = np.sqrt(np.sum((X_test_scaled - centroid) ** 2, axis=1))

in_ad = test_distances <= threshold
out_ad = test_distances > threshold

print(f"AD Threshold (95th percentile): {threshold:.2f}")
print(f"Test molecules INSIDE AD:  {in_ad.sum()} ({in_ad.sum()/len(in_ad)*100:.1f}%)")
print(f"Test molecules OUTSIDE AD: {out_ad.sum()} ({out_ad.sum()/len(out_ad)*100:.1f}%)")

**Q19. What percentage of test molecules is outside AD? Should predictions be trusted for those?**

Your answer: ...

### Step 18: Compare performance inside vs outside AD

In [ ]:
if in_ad.sum() > 0:
    r2_in = r2_score(y_test[in_ad], y_test_pred_rf[in_ad])
    rmse_in = np.sqrt(mean_squared_error(y_test[in_ad], y_test_pred_rf[in_ad]))
    print(f"Inside AD  — R² = {r2_in:.3f}, RMSE = {rmse_in:.3f}, n = {in_ad.sum()}")

if out_ad.sum() > 5:
    r2_out = r2_score(y_test[out_ad], y_test_pred_rf[out_ad])
    rmse_out = np.sqrt(mean_squared_error(y_test[out_ad], y_test_pred_rf[out_ad]))
    print(f"Outside AD — R² = {r2_out:.3f}, RMSE = {rmse_out:.3f}, n = {out_ad.sum()}")

**Q20. Is performance better inside AD than outside AD? What does this demonstrate?**

Your answer: ...

## Part 8 - Predicting New Molecules
### Step 19: Predict pIC50 for new molecules

In [ ]:
new_molecules = {
    'Donepezil': 'O=C(CC1CCN(CC1)Cc1ccccc1)c1cc2c(cc1)OCO2',
    'Galantamine': 'COc1ccc2c3c1OC1CC(O)C=CC31CCN(C)C2',
    'Rivastigmine': 'CCN(C)C(=O)Oc1cccc(c1)C(C)N(C)C',
    'Tacrine': 'Nc1c2c(nc3ccccc13)CCCC2',
    'Benzene': 'c1ccccc1',
    'Glucose': 'OCC1OC(O)C(O)C(O)C1O'
}

print(f"{'Molecule':<15} {'pIC50 (pred)':>12} {'IC50 (nM, pred)':>16} {'In AD?':>8}")
print('=' * 60)

for name, smi in new_molecules.items():
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print(f"{name:<15} INVALID SMILES")
        continue

    desc = compute_descriptors(smi)
    x_new = np.array([[desc[col] for col in feature_cols]])

    pred_pIC50 = rf_model.predict(x_new)[0]
    pred_ic50_nm = 10 ** (-pred_pIC50) * 1e9

    x_scaled = scaler.transform(x_new)
    dist = np.sqrt(np.sum((x_scaled - centroid) ** 2))
    ad_status = 'Yes' if dist <= threshold else 'NO!'

    print(f"{name:<15} {pred_pIC50:>12.2f} {pred_ic50_nm:>16.1f} {ad_status:>8}")

**Q21. Does the model predict high activity for known AChE inhibitors?**

Your answer: ...

**Q22. What does the model predict for Benzene and Glucose? Are they within AD?**

Your answer: ...

**Q23. Why is AD checking important before trusting predictions?**

Your answer: ...

## Part 9 - Wrap-Up & Reflection

**Q24. Summarise the complete QSAR pipeline you followed.**

Your answer: ...

**Q25. Main challenges or surprises during this practical session?**

Your answer: ...

**Q26. If you had more time, how would you improve this QSAR model? (at least 3 ideas)**

Your answer: ...

**Q27. For a QSAR model predicting cosmetic toxicity, what would you change in this pipeline?**

Your answer: ...

## Summary of all sessions

| Session | Focus | Tools | Duration |
|---|---|---|---|
| TP 1 | Databases, SMILES/InChI/InChIKey, data curation | PubChem, ChEMBL, DrugBank | ~3 h 30 |
| TP 2 | Python, RDKit, descriptors, fingerprints, Tanimoto | Google Colab / Jupyter | ~3 h 30 |
| TP 3 | Full QSAR pipeline and model evaluation | Google Colab / Jupyter | ~4 h 00 |

Good luck!